# Validate the GNN track matching between ND-LAr and TMS

Counterpart of `Plotting_validation_Mx2.ipynb` for the ND-LAr / TMS matching. The same checks are made on the GNN inference output
(`HDF5Reader` file of `validation_grappa_ndlar.cfg`), and every plot is plotly:

1. true vs predicted edges for every entry (spill) and a confusion matrix
2. edges split by detector pair (`NDLAr-NDLAr`, `NDLAr-TMS`, `TMS-TMS`) and by status (correct / missing / extra), with summary table and CSV
3. 3D spill display in mm: ND-LAr and TMS boxes (geometry of `SPINE_TMS_vertex_matching_viewer.ipynb`, from `TMS_Constants.h`),
   clusters coloured by true group, and the predicted edges drawn green (correct), red (extra) or dashed orange (missing true edge)
4. efficiency as a function of the number of true matches / clusters per spill

**Coordinates.** The GNN input (`create_LArCV_NDLAr_TMS_matched.py`) is voxelised over one box covering both detectors, in mm in the TMS frame.
`data['data']` therefore holds voxel indices; `voxel_to_mm` converts them back so everything is plotted in mm, like the viewer notebook.

In [ ]:
import sys

SOFTWARE_DIR = '/sdf/data/neutrino/software/spine/src/' # Change this path to your software install

# Set software directory
sys.path.append(SOFTWARE_DIR)

In [ ]:
import os, csv, math
from collections import defaultdict

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from IPython.display import display

from spine.io.core.read.hdf5 import HDF5Reader
from spine.utils.globals import GROUP_COL, COORD_COLS
from spine.utils.gnn.cluster import get_cluster_label

DATA_FILE = 'mcfiles/output/test_ndlar_nooverlap.h5'  # GNN inference output
PLOT_DIR = 'plots/validation_ndlar'
os.makedirs(PLOT_DIR, exist_ok=True)

reader = HDF5Reader(DATA_FILE)
print(f"{reader.num_entries} entries in {DATA_FILE}")

## Geometry

Voxel box = `make_larcv_box()` in `create_LArCV_NDLAr_TMS_matched.py` (mm, TMS frame); detector boxes = the ones drawn in
`SPINE_TMS_vertex_matching_viewer.ipynb` (active volumes from `TMS_Constants.h`).

In [ ]:
# Voxelisation box (mm) -- must match make_larcv_box() in create_LArCV_NDLAr_TMS_matched.py
BOX_MIN = np.array([-4000.0, -3705.0,  4000.0])
BOX_MAX = np.array([ 4000.0,  1000.0, 19000.0])
VOXEL_SIZE_MM = 3.72
BOX_NUM = np.ceil(np.abs(BOX_MAX - BOX_MIN) / VOXEL_SIZE_MM).astype(int)
BOX_SIZE = (BOX_MAX - BOX_MIN) / BOX_NUM  # actual voxel pitch per axis (LArCV Voxel3DMeta)

# Detector active volumes (mm), from TMS_Constants.h
NDLAR_X, NDLAR_Y, NDLAR_Z = (-3478.48, 3478.48), (-2166.71, 829.282), (4179.24, 9135.88)
TMS_X,   TMS_Y,   TMS_Z   = (-3730.0,  3730.0),  (-3702.23, 997.77),  (11177.5, 18502.5)

# The two detectors are separated along z; the gap midpoint splits clusters into volumes
Z_GAP_MM = 0.5 * (NDLAR_Z[1] + TMS_Z[0])


def voxel_to_mm(points):
    """(N, 3) voxel indices -> (N, 3) mm at the voxel centre."""
    return BOX_MIN + (np.asarray(points, dtype=float) + 0.5) * BOX_SIZE


print("voxel box:", BOX_NUM, "voxels, pitch [mm]:", np.round(BOX_SIZE, 3))
print(f"ND-LAr z [{NDLAR_Z[0]}, {NDLAR_Z[1]}] mm, TMS z [{TMS_Z[0]}, {TMS_Z[1]}] mm, gap midpoint {Z_GAP_MM:.1f} mm")

## Identify True & Predicted Track Matches for all entries

In [ ]:
number_clusters = []
label_mask = []
pred_mask = []
num_extra = []
num_missing = []
num_wrong = []
for entry in range(reader.num_entries):

    data = reader[entry]
    group_ids = get_cluster_label(data['data'], data['clusts'], column=GROUP_COL)
    number_clusters.append(len(group_ids))

    ###### True Match Cluster Segments ######
    # True if should be matched, false if shouldnt
    true_mask = (group_ids[data['edge_index'][:, 0]] == group_ids[data['edge_index'][:, 1]])
    label_mask.append(true_mask)
    ###### Predicted Matched Cluster Segments #######
    # True if GNN matched, false if GNN didn't match
    gnn_mask = data['edge_pred'][:, 1] > data['edge_pred'][:, 0]
    pred_mask.append(gnn_mask)

    set_true = set(map(tuple, data['edge_index'][true_mask]))
    set_pred = set(map(tuple, data['edge_index'][gnn_mask]))

    if len(set_true - set_pred) > 0:
        num_missing.append(entry)
    if len(set_pred - set_true) > 0:
        num_extra.append(entry)
    if len(set_true ^ set_pred) > 0:
        num_wrong.append(entry)

print(f"entries with missing matches: {len(num_missing)}, extra matches: {len(num_extra)}, any wrong: {len(num_wrong)}")

In [ ]:
## Takes some time ###
flattened_label = np.concatenate(label_mask).ravel().astype(bool)
flattened_pred = np.concatenate(pred_mask).ravel().astype(bool)

## Create "Confusion Matrix" Counting How Often It Is Right/Wrong

In [ ]:
import sklearn.metrics

con_mat = sklearn.metrics.confusion_matrix(flattened_label, flattened_pred)
labels = ["Not Connected", "Connected"]
sums = con_mat.sum(axis=0)  # percentages are per predicted-label column, as in the Mx2 notebook

def plot_confusion_matrix(con_mat, save=True):
    total = con_mat.sum()
    text = [[f"{con_mat[i, j]}<br>({100 * con_mat[i, j] / max(sums[j], 1):.1f}%)" for j in range(2)] for i in range(2)]
    fig = go.Figure(go.Heatmap(
        z=con_mat, x=labels, y=labels, colorscale="Blues", showscale=False,
        hovertemplate="True %{y}<br>Predicted %{x}<br>%{z}<extra></extra>"))
    for i in range(2):
        for j in range(2):
            # Dark cells get white text (same rule as the Mx2 notebook)
            fig.add_annotation(x=labels[j], y=labels[i], text=text[i][j], showarrow=False,
                               font=dict(size=18, color="white" if con_mat[i, j] > 0.7 * total else "black"))
    fig.update_layout(width=550, height=550, xaxis=dict(title="Predicted Label", side="bottom"),
                      yaxis=dict(title="True Label", autorange="reversed"),
                      font=dict(size=16), margin=dict(l=20, r=20, t=30, b=20))
    fig.show()
    if save:
        fig.write_html(f"{PLOT_DIR}/confusion_matrix.html", include_plotlyjs="cdn")
    return fig

fig_cm = plot_confusion_matrix(con_mat)

# Count issues by detector pair

Each cluster is classed as ND-LAr or TMS by which side of the z gap most of its points fall on.
Edges come in both directions, so they are folded to unordered pairs and the number of directions (0, 1, 2) that are true / predicted decides the status.

In [ ]:
Z_COL = COORD_COLS[2]  # the z column index in data['data']


def cluster_class(data_array, cluster_idx):
    """Classify a cluster as 'NDLAr', 'TMS' or 'other' from the mm z of its voxels."""
    z = voxel_to_mm(data_array[cluster_idx][:, COORD_COLS])[:, 2]
    in_ndlar = ((z >= NDLAR_Z[0]) & (z <= Z_GAP_MM)).sum()
    in_tms = ((z > Z_GAP_MM) & (z <= TMS_Z[1])).sum()
    if in_ndlar == 0 and in_tms == 0:
        return 'other'
    return 'NDLAr' if in_ndlar >= in_tms else 'TMS'


def edge_volume(ci, cj):
    if ci == 'NDLAr' and cj == 'NDLAr': return 'NDLAr-NDLAr'
    if ci == 'TMS' and cj == 'TMS':     return 'TMS-TMS'
    if {ci, cj} == {'NDLAr', 'TMS'}:    return 'NDLAr-TMS'
    return 'other'

In [ ]:
VOLUME_PAIRS = ['NDLAr-NDLAr', 'NDLAr-TMS', 'TMS-TMS', 'other']
STATUSES = ['correct_both', 'correct_one', 'missing', 'extra_both', 'extra_one']

counts = {v: {s: 0 for s in STATUSES} for v in VOLUME_PAIRS}
# entries[volume_pair][status] = list of (entry, cluster_i, cluster_j) tuples
entries = {v: {s: [] for s in STATUSES} for v in VOLUME_PAIRS}

asymmetric_graph_pairs = 0
graph_pairs = np.zeros(reader.num_entries)  # unordered cluster pairs present in the GNN graph, per entry

for entry in range(reader.num_entries):
    data = reader[entry]
    edges = data['edge_index']
    if len(edges) == 0:
        continue

    group_ids = get_cluster_label(data['data'], data['clusts'], column=GROUP_COL)
    true_mask = group_ids[edges[:, 0]] == group_ids[edges[:, 1]]
    pred_mask_e = data['edge_pred'][:, 1] > data['edge_pred'][:, 0]

    klass = [cluster_class(data['data'], c) for c in data['clusts']]

    pair_info = defaultdict(lambda: {'truth': 0, 'pred': 0, 'graph': 0, 'vol': None, 'i': None, 'j': None})
    for k, (i, j) in enumerate(edges):
        info = pair_info[frozenset((int(i), int(j)))]
        info['truth'] += int(true_mask[k])
        info['pred'] += int(pred_mask_e[k])
        info['graph'] += 1
        if info['vol'] is None:
            info['vol'] = edge_volume(klass[i], klass[j])
            info['i'], info['j'] = int(i), int(j)

    graph_pairs[entry] = len(pair_info)
    for info in pair_info.values():
        if info['graph'] < 2:
            asymmetric_graph_pairs += 1
        t, p, vol = info['truth'], info['pred'], info['vol']
        status = None
        if   t == 2 and p == 2: status = 'correct_both'
        elif t == 2 and p == 1: status = 'correct_one'
        elif t == 2 and p == 0: status = 'missing'
        elif t == 0 and p == 2: status = 'extra_both'
        elif t == 0 and p == 1: status = 'extra_one'

        if status is not None:
            counts[vol][status] += 1
            entries[vol][status].append((entry, info['i'], info['j']))

if asymmetric_graph_pairs:
    print(f"Note: {asymmetric_graph_pairs} pair(s) had only one direction in edge_index.")

In [ ]:
def summary_table(counts=counts):
    rows = []
    for v in VOLUME_PAIRS:
        cb, co, m = counts[v]['correct_both'], counts[v]['correct_one'], counts[v]['missing']
        eb, eo = counts[v]['extra_both'], counts[v]['extra_one']
        truth_total = cb + co + m
        tp_dir, fp_dir = 2 * cb + co, 2 * eb + eo
        rows.append({
            'volume_pair': v, 'correct_both': cb, 'correct_one': co, 'missing': m,
            'extra_both': eb, 'extra_one': eo, 'truth_total': truth_total,
            'efficiency_full': cb / truth_total if truth_total else float('nan'),
            'efficiency_any': (cb + co) / truth_total if truth_total else float('nan'),
            'purity_directed': tp_dir / (tp_dir + fp_dir) if (tp_dir + fp_dir) else float('nan'),
        })
    return pd.DataFrame(rows)


def entry_summary_table(entry):
    '''Same table for one entry (spill) only.'''
    return summary_table({v: {s: sum(e == entry for e, _, _ in entries[v][s]) for s in STATUSES}
                          for v in VOLUME_PAIRS})


summary_df = summary_table()
summary_df.style.format({'efficiency_full': '{:.3f}', 'efficiency_any': '{:.3f}', 'purity_directed': '{:.3f}'}, na_rep='-').hide(axis='index')

In [ ]:
TXT_PATH = f"{PLOT_DIR}/volume_pair_summary.txt"
CSV_PATH = f"{PLOT_DIR}/volume_pair_summary.csv"

with open(TXT_PATH, 'w') as f:
    f.write("Volume-pair edge-classification summary (ND-LAr / TMS)\n")
    f.write(f"File: {DATA_FILE}\n")
    f.write(f"Entries processed: {reader.num_entries}\n\n")
    f.write(summary_df.to_string(index=False, float_format=lambda x: f"{x:.3f}", na_rep='-') + "\n")
    f.write("\nLegend:\n")
    f.write("  correct_both : true pair, GNN predicted both directions\n")
    f.write("  correct_one  : true pair, GNN predicted only one direction\n")
    f.write("  missing      : true pair, GNN predicted neither direction\n")
    f.write("  extra_both   : non-true pair, GNN predicted both directions\n")
    f.write("  extra_one    : non-true pair, GNN predicted one direction\n")
    f.write("  truth_total  : correct_both + correct_one + missing\n")
    f.write("  efficiency_full : correct_both / truth_total\n")
    f.write("  efficiency_any  : (correct_both + correct_one) / truth_total\n")
    f.write("  purity_directed : (2*cb + co) / (2*cb + co + 2*eb + eo)\n")
summary_df.to_csv(CSV_PATH, index=False)

print(f"Wrote {TXT_PATH}")
print(f"Wrote {CSV_PATH}")

In [ ]:
# Stacked bar of the five statuses per detector pair
fig = go.Figure()
for s, colour in zip(STATUSES, ['#009E73', '#56B4E9', '#E69F00', '#D55E00', '#CC79A7']):
    fig.add_trace(go.Bar(x=VOLUME_PAIRS, y=[counts[v][s] for v in VOLUME_PAIRS], name=s, marker_color=colour))
fig.update_layout(barmode='stack', xaxis_title="Detector pair", yaxis_title="Number of cluster pairs",
                  title="Edge status by detector pair", font=dict(size=14), width=800, height=450)
fig.show()
fig.write_html(f"{PLOT_DIR}/volume_pair_status.html", include_plotlyjs="cdn")

In [ ]:
# --- per-bucket entry listing ---
def show(volume_pair, status, max_entries=None, group_by_entry=True):
    """Print events that fall into a given (volume_pair, status) bucket.

    volume_pair : one of 'NDLAr-NDLAr', 'NDLAr-TMS', 'TMS-TMS', 'other'
    status      : one of 'correct_both', 'correct_one', 'missing', 'extra_both', 'extra_one'
    max_entries : cap on number of events shown (None = all)
    group_by_entry : if True, list each entry once with all its pairs
    """
    items = entries[volume_pair][status]
    print(f"\n[{volume_pair} / {status}]  {len(items)} pair(s) "
          f"across {len(set(e for e, _, _ in items))} event(s)")
    if not items:
        return

    if group_by_entry:
        by_entry = defaultdict(list)
        for e, i, j in items:
            by_entry[e].append((i, j))
        shown = 0
        for e in sorted(by_entry):
            if max_entries is not None and shown >= max_entries:
                print(f"  ... ({len(by_entry) - shown} more events not shown)")
                break
            print(f"  entry {e}: " + ', '.join(f"({i},{j})" for i, j in by_entry[e]))
            shown += 1
    else:
        for e, i, j in items[:max_entries]:
            print(f"  entry {e}: clusters ({i}, {j})")
        if max_entries is not None and len(items) > max_entries:
            print(f"  ... ({len(items) - max_entries} more not shown)")


# Quick example: show first 10 entries of every non-empty bucket
print("\n=== bucket contents ===")
for v in VOLUME_PAIRS:
    for s in STATUSES:
        if counts[v][s] > 0:
            show(v, s, max_entries=10)

In [ ]:
show("NDLAr-TMS", "correct_both", max_entries=1000)

## Plot One Spill Display

Clusters are drawn in mm in the TMS frame, with the ND-LAr and TMS boxes from the viewer notebook.

### Detector boxes and figure setup (from `SPINE_TMS_vertex_matching_viewer.ipynb`)

In [ ]:
def plot_box(fig, x_min, x_max, y_min, y_max, z_min, z_max, color, name):
    # One continuous path over every edge; None breaks the line between disconnected edges
    x = [x_min, x_max, x_max, x_min, x_min, x_min, x_max, x_max, x_min, x_min, None, x_max, x_max, None, x_min, x_min, None, x_max, x_max]
    y = [y_min, y_min, y_max, y_max, y_min, y_min, y_min, y_max, y_max, y_min, None, y_min, y_min, None, y_max, y_max, None, y_max, y_max]
    z = [z_min, z_min, z_min, z_min, z_min, z_max, z_max, z_max, z_max, z_max, None, z_min, z_max, None, z_min, z_max, None, z_min, z_max]
    fig.add_trace(go.Scatter3d(x=x, y=y, z=z, mode="lines",
                               line=dict(color=color, width=3), name=name, showlegend=True))


def draw_NDLAr(fig):
    plot_box(fig, *NDLAR_X, *NDLAR_Y, *NDLAR_Z, "black", "ND-LAr detector")


def draw_TMS(fig):
    plot_box(fig, *TMS_X, *TMS_Y, *TMS_Z, "blue", "TMS detector")


def setup_figure(fig, title):
    fig.update_layout(
        width=1600, height=900,
        scene=dict(
            xaxis=dict(title="x [mm]", range=[-4500, 4500]),
            yaxis=dict(title="y [mm]", range=[-4000, 1000]),
            zaxis=dict(title="z [mm]", range=[0, 20000]),
            camera=dict(eye=dict(x=2.0, y=4.0, z=0.0), up=dict(x=0, y=-1, z=0)),  # Y is up
            aspectmode='manual',
            aspectratio=dict(x=1, y=1, z=4),
        ),
        margin=dict(l=0, r=0, b=0, t=50),
        showlegend=True,
        legend=dict(x=0.02, y=0.98, xanchor='left', yanchor='top',
                    bgcolor='rgba(255, 255, 255, 0.8)', bordercolor='Black', borderwidth=1,
                    itemsizing='constant'),
        title=dict(text=title, x=0.5, xanchor='center'),
    )
    fig.data = []
    draw_TMS(fig)
    draw_NDLAr(fig)
    return fig

### Plotting clusters and matches

`plot_3D_viewer(entry, plot_groups=None, show_tp=True, show_fp=True, show_fn=True, show_unique=True)` draws each cluster as markers coloured by its true group (same colour = should be matched), then the edges between clusters:
- **green**: true pair, predicted (true positive)
- **red**: predicted but not true (false positive)
- **dashed orange**: true but not predicted (false negative)

Edges between ND-LAr and TMS clusters join the facing boundary points (most downstream ND-LAr point to most upstream TMS point) so the line crosses the gap cleanly; all other edges join the closest pair of points.

**Buttons** (top): on/off for the green lines, red lines, dashed orange lines, and the *unique fragments* (fragments whose group ID no other fragment shares). All start off; `show_*` set the starting state. Below them, a filter on the match lines by detector pair: *All pairs* (filter off), *NDLAr-NDLAr only*, *NDLAr-TMS only* or *TMS-TMS only* (one at a time; `pair_filter=` sets the start).

Match lines are 2 px wide (`line_width`). The plot is `width=1000` px wide (`plot_height=600`); lower `width` if it still goes off the page. After the plot, the status table of that spill is shown (`show_summary=False` to skip).

**Sliders** (bottom): one each for true positive, false positive and false negative groups, **cumulative**: sliding right adds more groups. A group brings *all* of its fragments, even if only 2 of 3 were matched correctly. A false positive pair adds the groups of both its fragments. A match line appears once the groups of both its fragments are in. The three sliders are independent of each other and of the buttons.

`plot_groups=[3, 6, 90]` makes those the only positions of every slider (number of groups shown; clipped to the groups available in that category). With `None` the sliders step through every count from 0 to the number of groups.

In [ ]:
CB_color_cycle = ['#332288', '#117733', '#44AA99', '#88CCEE', '#EA4040', '#0B57CB', '#D79ABB', '#7D7A7A', '#D0B58F']
EDGE_COLOURS = {'correct': '#009E73', 'extra': '#d62728', 'missing': '#E69F00'}


def closest_points(a, b):
    """Indices of the closest pair of points between two (N, 3) / (M, 3) arrays (chunked brute force)."""
    best = (np.inf, 0, 0)
    for s in range(0, len(a), 512):
        d = np.linalg.norm(a[s:s + 512, None, :] - b[None, :, :], axis=2)
        k = np.unravel_index(d.argmin(), d.shape)
        if d[k] < best[0]:
            best = (d[k], s + k[0], k[1])
    return best[1], best[2]


def edge_endpoints(pts_i, pts_j, class_i, class_j):
    """Line endpoints (mm) joining two clusters."""
    if {class_i, class_j} == {'NDLAr', 'TMS'}:
        nd, tms = (pts_i, pts_j) if class_i == 'NDLAr' else (pts_j, pts_i)
        p_nd, p_tms = nd[np.argmax(nd[:, 2])], tms[np.argmin(tms[:, 2])]
        return (p_nd, p_tms) if class_i == 'NDLAr' else (p_tms, p_nd)
    a, b = closest_points(pts_i, pts_j)
    return pts_i[a], pts_j[b]


def edge_status(true_set, pred_set):
    """{unordered pair: 'correct' | 'extra' | 'missing'} from sets of (i, j) tuples."""
    status = {}
    for i, j in true_set | pred_set:
        key = tuple(sorted((int(i), int(j))))
        t = (i, j) in true_set or (j, i) in true_set
        p = (i, j) in pred_set or (j, i) in pred_set
        status[key] = 'correct' if t and p else ('missing' if t else 'extra')
    return status


SLIDER_ROW_PX = 90  # vertical space reserved for each slider (value text + bar + step labels)


def plot_3D_viewer(entry, plot_groups=None, show_tp=False, show_fp=False, show_fn=False, show_unique=False,
                   pair_filter='all', markersize=2, width=1000, plot_height=600, line_width=2,
                   show_summary=True, save_html=None, return_fig=False):
    '''Plot one spill with on/off buttons and cumulative group sliders.

    plot_groups : list of numbers of groups, e.g. [3, 6, 90]. These are the only positions of each slider
                  (a value larger than the groups available is clipped to all of them). None: every count 0..N.
    show_tp / show_fp / show_fn / show_unique : starting state of the line / fragment buttons (default off)
    pair_filter : starting choice of the detector-pair filter: 'all', 'NDLAr-NDLAr', 'NDLAr-TMS' or 'TMS-TMS'
    width, plot_height : size of the plot in pixels (width counts the margins); make width smaller if it goes off the page
    show_summary : print the status table of this spill after the plot
    '''
    print("Entry ", entry)
    data = reader[entry]

    # Predicted matches
    mask = data['edge_pred'][:, 1] > data['edge_pred'][:, 0]
    edge_index = data['edge_index'][mask]
    print("Predicted connection:")
    print(edge_index)

    # True matches
    group_ids = get_cluster_label(data['data'], data['clusts'], column=GROUP_COL)
    true_mask = (group_ids[data['edge_index'][:, 0]] == group_ids[data['edge_index'][:, 1]])
    real = data['edge_index'][true_mask]
    print("True connection:")
    print(real)

    gid = lambda c: int(group_ids[c])
    pts = [voxel_to_mm(data['data'][clust][:, COORD_COLS]) for clust in data['clusts']]
    klass = [cluster_class(data['data'], clust) for clust in data['clusts']]
    members = defaultdict(list)  # group id -> cluster indices
    group_colour = {}
    for c in range(len(pts)):
        members[gid(c)].append(c)
        group_colour.setdefault(gid(c), CB_color_cycle[len(group_colour) % len(CB_color_cycle)])

    # Unordered pairs by status: tp = correct, fp = extra, fn = missing
    status = edge_status(set(map(tuple, real)), set(map(tuple, edge_index)))
    cat_status = {'tp': 'correct', 'fp': 'extra', 'fn': 'missing'}
    cat_edges = {k: sorted(e for e, st in status.items() if st == s) for k, s in cat_status.items()}
    # Groups each slider adds, in order of first appearance. A group brings all of its fragments, even those
    # not in the pair (3 fragments, 2 matched correctly: the third still comes with the group).
    cat_groups = {}
    for k, edges in cat_edges.items():
        order = []
        for i, j in edges:
            for g in (gid(i), gid(j)):
                if g not in order:
                    order.append(g)
        cat_groups[k] = order
    print({k: f"{len(cat_edges[k])} pairs in {len(v)} groups" for k, v in cat_groups.items()})

    fig = go.Figure()
    setup_figure(fig, f"Entry {entry}: GNN matches between ND-LAr and TMS clusters")

    def add_fragment(c, visible=True, opacity=1.0):
        fig.add_trace(go.Scatter3d(
            x=pts[c][:, 0], y=pts[c][:, 1], z=pts[c][:, 2], mode='markers', visible=visible, opacity=opacity,
            marker=dict(size=markersize, color=group_colour[gid(c)]), showlegend=False,
            name=f"cluster {c} (group {gid(c)}, {klass[c]}, {len(pts[c])} voxels)", hoverlabel_namelength=-1))
        return len(fig.data) - 1

    # Slider steps: how many groups of each category are shown
    def step_counts(n_avail):
        return list(range(n_avail + 1)) if plot_groups is None else [min(int(v), n_avail) for v in plot_groups]

    slider_traces = {k: [] for k in cat_status}  # (trace index, groups needed before it shows)
    edge_idx = {k: [] for k in cat_status}       # edge traces per category, for the buttons
    edge_pair = {}                               # edge trace index -> detector pair of its two fragments
    start_on = {'tp': show_tp, 'fp': show_fp, 'fn': show_fn}
    first_shown = {k: step_counts(len(cat_groups[k]))[0] for k in cat_status}

    for k in cat_status:
        rank = {g: r for r, g in enumerate(cat_groups[k])}
        # Fragments of every group this slider can add (one copy per slider, so the sliders do not
        # fight over the visibility of the same trace)
        for g in cat_groups[k]:
            for c in members[g]:
                slider_traces[k].append((add_fragment(c, visible=rank[g] < first_shown[k]), rank[g] + 1))
        # Match lines: shown once the groups of both fragments are in
        for i, j in cat_edges[k]:
            a, b = edge_endpoints(pts[i], pts[j], klass[i], klass[j])
            need = max(rank[gid(i)], rank[gid(j)]) + 1
            fig.add_trace(go.Scatter3d(
                x=[a[0], b[0]], y=[a[1], b[1]], z=[a[2], b[2]], mode='lines',
                visible=need <= first_shown[k], opacity=1.0 if start_on[k] else 0.0,
                line=dict(color=EDGE_COLOURS[cat_status[k]], width=line_width, dash='dash' if k == 'fn' else 'solid'),
                showlegend=False, name=f"{cat_status[k]} match: clusters {i}-{j}", hoverlabel_namelength=-1))
            slider_traces[k].append((len(fig.data) - 1, need))
            edge_idx[k].append(len(fig.data) - 1)
            edge_pair[len(fig.data) - 1] = edge_volume(klass[i], klass[j])

    # Fragments sharing their group id with no other fragment
    unique_idx = [add_fragment(c, opacity=1.0 if show_unique else 0.0)
                  for c in range(len(pts)) if len(members[gid(c)]) == 1]

    # Buttons: the on/off buttons change opacity, the detector-pair filter changes line width and the
    # sliders change visibility, so they all combine
    def toggle_menu(n, label, idx, on_now):
        on, off = [{'opacity': 1.0}, idx], [{'opacity': 0.0}, idx]
        return dict(type='buttons', direction='right', showactive=True, active=-1,
                    x=0.0 + 0.25 * n, xanchor='left', y=1.0, yanchor='bottom', pad=dict(t=4, b=4),
                    buttons=[dict(label=f"{label} on/off", method='restyle',
                                  args=off if on_now else on, args2=on if on_now else off)])

    toggles = [('tp', "True positives (green)", edge_idx['tp'], show_tp),
               ('fp', "False positives (red)", edge_idx['fp'], show_fp),
               ('fn', "False negatives (dashed orange)", edge_idx['fn'], show_fn),
               ('unique', "Unique fragments", unique_idx, show_unique)]
    menus = [toggle_menu(n, label, idx, on) for n, (_, label, idx, on) in enumerate(t for t in toggles if t[2])]

    # Detector-pair filter on the match lines: one choice at a time, "All pairs" = filter off
    pair_options = [('all', "All pairs"), ('NDLAr-NDLAr', "NDLAr-NDLAr only"),
                    ('NDLAr-TMS', "NDLAr-TMS only"), ('TMS-TMS', "TMS-TMS only")]
    assert pair_filter in [m for m, _ in pair_options], f"pair_filter must be one of {pair_options}"
    all_edges = sorted(edge_pair)

    def pair_widths(mode):
        return [line_width if mode == 'all' or edge_pair[i] == mode else 0 for i in all_edges]

    for i in all_edges:
        fig.data[i].line.width = pair_widths(pair_filter)[all_edges.index(i)]
    if all_edges:
        menus.append(dict(type='buttons', direction='right', showactive=True,
                          active=[m for m, _ in pair_options].index(pair_filter),
                          x=0.0, xanchor='left', y=1.09, yanchor='bottom', pad=dict(t=4, b=4),
                          buttons=[dict(label=label, method='restyle', args=[{'line.width': pair_widths(m)}, all_edges])
                                   for m, label in pair_options]))

    # Sliders: cumulative groups for each category
    slider_names = {'tp': "True positive groups", 'fp': "False positive groups", 'fn': "False negative groups"}
    sliders = []
    for k in cat_status:
        if not cat_groups[k]:
            continue
        idx = [i for i, _ in slider_traces[k]]
        counts = step_counts(len(cat_groups[k]))
        labels = [str(n) for n in counts] if plot_groups is None else [str(v) for v in plot_groups]
        steps = [dict(label=lab, method='restyle',
                      args=[{'visible': [n >= need for _, need in slider_traces[k]]}, idx])
                 for lab, n in zip(labels, counts)]
        sliders.append(dict(active=0, steps=steps, x=0.15, len=0.75, xanchor='left',
                            y=-(15 + SLIDER_ROW_PX * len(sliders)) / plot_height, yanchor='top', pad=dict(t=0, b=0),
                            currentvalue=dict(prefix=f"{slider_names[k]} (of {len(cat_groups[k])}): ")))

    # Pixel layout: the sliders sit under the plot, SLIDER_ROW_PX apart, so they cannot overlap
    margin = dict(l=0, r=0, t=130, b=20 + SLIDER_ROW_PX * len(sliders))
    fig.update_layout(updatemenus=menus, sliders=sliders, margin=margin, width=width,
                      height=plot_height + margin['t'] + margin['b'])
    fig.show()
    if show_summary:
        print(f"Summary for entry {entry}")
        display(entry_summary_table(entry).style.format(
            {'efficiency_full': '{:.3f}', 'efficiency_any': '{:.3f}', 'purity_directed': '{:.3f}'}, na_rep='-').hide(axis='index'))
    if save_html:
        fig.write_html(save_html, include_plotlyjs=True)
        print(f"Saved {save_html}")
    if return_fig:
        return fig

In [ ]:
num_wrong[:10]

In [ ]:
event_id = num_wrong[0] if num_wrong else 0
fig = plot_3D_viewer(event_id, plot_groups=[3, 6, 90], return_fig=True)
fig.write_html(f"{PLOT_DIR}/validation_event_display{event_id}.html", include_plotlyjs="cdn")
# fig.write_image(f"{PLOT_DIR}/validation_event_display{event_id}.png")  # needs kaleido

## Efficiency as a function of tracks

In [ ]:
# Number of true / predicted matches per spill (edges are in both directions, so divide by 2)
matches = np.array([np.sum(arr) / 2. for arr in label_mask])
pred_matches = np.array([np.sum(arr) / 2. for arr in pred_mask])
num_clust = np.array(number_clusters)

mask_indices = np.zeros(len(matches), dtype=bool)
mask_indices[num_wrong] = True

print("Largest numbers of true matches per spill:", np.sort(matches)[-10:])
print("Largest numbers of clusters per spill:", np.sort(num_clust)[-10:])

In [ ]:
def hist_with_wrong(values, wrong_mask, xmax, xlabel, title):
    """All spills vs spills with a wrong prediction, one count per integer bin, and the fraction wrong."""
    bins = np.arange(0, xmax + 1)
    all_counts, _ = np.histogram(values, bins=bins)
    wrong_counts, _ = np.histogram(values[wrong_mask], bins=bins)
    frac = np.divide(wrong_counts, all_counts, out=np.full(len(all_counts), np.nan), where=all_counts > 0)
    x = bins[:-1]

    fig = go.Figure()
    fig.add_trace(go.Bar(x=x, y=all_counts, name="All spills", marker_color='#0072B2', opacity=0.7))
    fig.add_trace(go.Bar(x=x, y=wrong_counts, name="Prediction wrong", marker_color='#D55E00', opacity=0.8))
    fig.update_layout(barmode='overlay', xaxis_title=xlabel, yaxis_title="Number of spills", title=title,
                      font=dict(size=14), width=800, height=450)
    fig.show()

    fig_frac = go.Figure(go.Scatter(x=x, y=frac, mode='lines+markers', name="Fraction wrong",
                                    line=dict(color='#0072B2'), marker=dict(size=7)))
    fig_frac.update_layout(xaxis_title=xlabel, yaxis_title="Fraction of spills with a wrong prediction",
                           title=title.replace("Number", "Fraction wrong vs number"),
                           font=dict(size=14), width=800, height=450)
    fig_frac.show()
    return fig, fig_frac, frac

In [ ]:
fig_m, fig_m_frac, wrong_per_true_match = hist_with_wrong(
    matches, mask_indices, 8, "Number matches per spill", "Number total matches per spill given to GNN")
print(wrong_per_true_match)

In [ ]:
fig_c, fig_c_frac, wrong_per_true_particles = hist_with_wrong(
    num_clust, mask_indices, 36, "Number clusters per spill", "Number total clusters per spill given to GNN")

In [ ]:
fig = go.Figure(go.Histogram2d(
    x=num_clust, y=matches, xbins=dict(start=0, end=35, size=1), ybins=dict(start=0, end=7, size=1),
    colorscale='Viridis', colorbar=dict(title="Spills")))
fig.update_layout(xaxis_title="Number clusters per spill", yaxis_title="Number true matches per spill",
                  title="True matches vs. number of clusters per spill", font=dict(size=14), width=800, height=500)
fig.show()

In [ ]:
# Predicted vs true matches (log colour scale, empty bins left blank)
bins = np.arange(0, 8)
h, _, _ = np.histogram2d(matches, pred_matches, bins=(np.arange(0, 8), np.arange(0, 8)))
fig = go.Figure(go.Heatmap(x=bins[:-1], y=bins[:-1], z=np.log10(np.where(h > 0, h, np.nan)).T,
                           customdata=h.T, hovertemplate="True %{x}<br>Predicted %{y}<br>%{customdata:.0f} spills<extra></extra>",
                           colorscale='Viridis', colorbar=dict(title="log10(spills)")))
fig.add_trace(go.Scatter(x=[0, 6], y=[0, 6], mode='lines', line=dict(color='white', dash='dot'), showlegend=False))
fig.update_layout(xaxis_title="Number True Matches", yaxis_title="Number Predicted Matches",
                  title="Number of Predicted vs. True Matches", font=dict(size=14), width=700, height=600)
fig.show()

In [ ]:
h, _, _ = np.histogram2d(num_clust, pred_matches, bins=(np.arange(0, 41), np.arange(0, 8)))
fig = go.Figure(go.Heatmap(x=np.arange(0, 40), y=np.arange(0, 7), z=np.log10(np.where(h > 0, h, np.nan)).T,
                           customdata=h.T, hovertemplate="Clusters %{x}<br>Predicted %{y}<br>%{customdata:.0f} spills<extra></extra>",
                           colorscale='Viridis', colorbar=dict(title="log10(spills)")))
fig.update_layout(xaxis_title="Number Clusters", yaxis_title="Number Predicted Matches",
                  title="Number of Predicted Matches vs. Number of Clusters", font=dict(size=14), width=800, height=500)
fig.show()

## Connections per spill by detector pair

A connection is a pair of clusters (fragments), and it only counts if **both directions** of the edge say so:
- **true connection**: both directions are true (same group id)
- **true positive connection**: true, and the GNN predicted both directions (`correct_both`)

A pair with only one direction predicted (`correct_one`) therefore counts as *not* connected, i.e. it is a true connection that is not a true positive.

In [ ]:
n_spills = reader.num_entries
true_conn = {v: np.zeros(n_spills, dtype=int) for v in VOLUME_PAIRS}  # true connections per spill
tp_conn = {v: np.zeros(n_spills, dtype=int) for v in VOLUME_PAIRS}    # true positive connections per spill
for v in VOLUME_PAIRS:
    for s in ('correct_both', 'correct_one', 'missing'):  # truth == 2 directions
        for e, _, _ in entries[v][s]:
            true_conn[v][e] += 1
    for e, _, _ in entries[v]['correct_both']:             # both directions predicted
        tp_conn[v][e] += 1

In [ ]:
def hist_true_vs_tp(true_counts, tp_counts, pair_name):
    """Overlaid integer-bin histograms of the true and the true positive connections per spill."""
    # One bar per value that occurs: x = the value per spill, y = how many spills have that value
    true_vals, true_n = np.unique(true_counts, return_counts=True)
    tp_vals, tp_n = np.unique(tp_counts, return_counts=True)
    fig = go.Figure()
    fig.add_trace(go.Bar(x=true_vals, y=true_n, name="True connections", marker_color='#0072B2', opacity=0.7))
    fig.add_trace(go.Bar(x=tp_vals, y=tp_n, name="True positive connections", marker_color='#009E73', opacity=0.7))
    lo = min(true_vals.min(), tp_vals.min()); hi = max(true_vals.max(), tp_vals.max())  # x axis need not start at 0
    fig.update_layout(barmode='overlay', xaxis=dict(title=f"Number of {pair_name} connections in a spill", range=[lo - 1, hi + 1]),
                      yaxis_title="Number of spills with this value", title=f"{pair_name} connections per spill (both directions required)",
                      font=dict(size=14), width=800, height=450)
    fig.show()
    fig.write_html(f"{PLOT_DIR}/connections_per_spill_{pair_name}.html", include_plotlyjs="cdn")
    print(f"{pair_name}: {true_counts.sum()} true connections, {tp_counts.sum()} true positive "
          f"({tp_counts.sum() / max(true_counts.sum(), 1):.3f})")
    return fig

fig_nn = hist_true_vs_tp(true_conn['NDLAr-NDLAr'], tp_conn['NDLAr-NDLAr'], "NDLAr-NDLAr")

In [ ]:
fig_nt = hist_true_vs_tp(true_conn['NDLAr-TMS'], tp_conn['NDLAr-TMS'], "NDLAr-TMS")

In [ ]:
fig_tt = hist_true_vs_tp(true_conn['TMS-TMS'], tp_conn['TMS-TMS'], "TMS-TMS")

### True connections / all possible connections per spill

All possible connections = every pair of clusters in the spill, `n (n - 1) / 2`. Set `DENOMINATOR = 'graph_edges'` to divide by
the pairs actually present in the GNN graph instead (the two agree if the graph is fully connected). Spills with no possible connection (fewer than 2 clusters) are kept and count as a fraction of 0, so they fall in the first bin.

In [ ]:
DENOMINATOR = 'clusters'  # 'clusters': all cluster pairs, 'graph_edges': pairs present in the GNN graph

total_true = sum(true_conn[v] for v in VOLUME_PAIRS)
if DENOMINATOR == 'clusters':
    possible = num_clust * (num_clust - 1) / 2
else:
    possible = graph_pairs
# Spills with no possible connection (0 or 1 cluster) are kept, with a fraction of 0: nothing to connect
fraction_true = np.divide(total_true, possible, out=np.zeros(len(possible)), where=possible > 0)
n_empty = int((possible == 0).sum())

fig = go.Figure(go.Histogram(x=fraction_true, xbins=dict(start=0, end=1, size=0.05), marker_color='#0072B2'))
fig.update_layout(xaxis=dict(title="True connections / all possible connections in a spill",
                             range=[fraction_true.min() - 0.05, fraction_true.max() + 0.05]),
                  yaxis_title="Number of spills with this value",
                  title=f"Fraction of possible connections that are true ({DENOMINATOR}; {len(fraction_true)} spills)",
                  font=dict(size=14), width=800, height=450)
fig.show()
fig.write_html(f"{PLOT_DIR}/true_over_possible_connections.html", include_plotlyjs="cdn")
print(f"{n_empty} spills have no possible connection (counted in the first bin)")
print(f"mean fraction {fraction_true.mean():.4f}, max {fraction_true.max():.4f}")